# One-vs-Rest role detectors

Four independent binary classifiers, one per legal role, each trained to distinguish its role from the other three. Unlike Stage 2, a sentence can be flagged positive by more than one detector, or none.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TEST_DF_PATH = Path("predictions/test_df.csv")
REGISTRY_PATH = Path("roberta_models/best_models_registry.json")
MODELS_DIR = Path("roberta_models")

TEXT_COL = "sent_text"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LEGAL_LABELS = ["beoordeling", "beslissing", "materiele feiten", "proceshandelingen"]
OVR_MODEL_KEYS = ["ovr_beoordeling", "ovr_beslissing", "ovr_materiele_feiten", "ovr_proceshandelingen"]


def clean_col(label):
    return label.strip().lower().replace(" ", "_")


print("Device:", DEVICE)

Device: cuda


## 2. Load test data

In [2]:
test_df = pd.read_csv(TEST_DF_PATH, keep_default_na=False)

print("Test rows:", len(test_df))
print("Gold-relevant rows for evaluation:", test_df["label"].isin(LEGAL_LABELS).sum())
test_df["label"].value_counts()

Test rows: 1502
Gold-relevant rows for evaluation: 961


label
None                 541
beoordeling          389
materiele feiten     297
proceshandelingen    206
beslissing            69
Name: count, dtype: int64

## 3. Load the 4 OvR models and run inference

In [3]:
with open(REGISTRY_PATH, "r", encoding="utf-8") as f:
    registry = json.load(f)


def load_model(model_key):
    model_info = registry[model_key]
    model_path = MODELS_DIR / Path(model_info["saved_path"]).name

    if not model_path.exists():
        raise FileNotFoundError(f"Model path not found: {model_path}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(DEVICE)
    model.eval()

    max_len = model_info.get("max_len", 256)
    return tokenizer, model, max_len


@torch.no_grad()
def predict_probs(texts, tokenizer, model, max_len, batch_size=32, desc=""):
    all_probs = []

    for start in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[start:start + batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt"
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}

        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=-1)
        all_probs.append(probs.cpu().numpy())

    return np.vstack(all_probs)


texts = test_df[TEXT_COL].fillna("").astype(str).tolist()

for model_key in OVR_MODEL_KEYS:
    tokenizer, model, max_len = load_model(model_key)
    probs = predict_probs(texts, tokenizer, model, max_len, BATCH_SIZE, desc=model_key)

    pos_label = clean_col(registry[model_key]["pos_label"])
    test_df[f"{pos_label}_p_not"] = probs[:, 0]
    test_df[f"{pos_label}_p_pos"] = probs[:, 1]
    test_df[f"{pos_label}_pred"] = (probs[:, 1] > probs[:, 0]).astype(int)

    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

print("OvR detectors run:", OVR_MODEL_KEYS)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_beoordeling:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_beslissing:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_materiele_feiten:   0%|          | 0/47 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

ovr_proceshandelingen:   0%|          | 0/47 [00:00<?, ?it/s]

OvR detectors run: ['ovr_beoordeling', 'ovr_beslissing', 'ovr_materiele_feiten', 'ovr_proceshandelingen']


## 4. Save raw model outputs

In [4]:
OUTPUT_PATH = Path("predictions/ovr_detectors_inference.csv")
test_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_PATH)

Saved: predictions\ovr_detectors_inference.csv


## 5. Evaluate

Positive-class metrics per detector, evaluated on the 961 gold-relevant sentences (each detector was trained only on that subset). Sanity check: should reproduce the previously reported OvR numbers (macro P 0.795, macro R 0.728, macro F1 0.758) — this is the positive-class-only average, matching Table 2/3/6 of the paper, not the both-classes-averaged "OvR mean" definition used elsewhere (see the earlier flagged inconsistency between those two).

In [5]:
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support

legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()
print("Rows used for evaluation:", len(legal_df))

results = []

for label in LEGAL_LABELS:
    clean = clean_col(label)

    y_true = (legal_df["label"] == label).astype(int)
    y_pred = legal_df[f"{clean}_pred"].astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )

    results.append({
        "label": label, "precision": round(precision, 4), "recall": round(recall, 4),
        "f1": round(f1, 4), "support": int(y_true.sum())
    })

    print(f"\n{'=' * 70}\nOvR: {label}\n{'=' * 70}")
    print(classification_report(
        y_true, y_pred, labels=[0, 1], target_names=[f"not_{label}", label], digits=4, zero_division=0
    ))

results_df = pd.DataFrame(results)
macro_p = results_df["precision"].mean()
macro_r = results_df["recall"].mean()
macro_f1 = results_df["f1"].mean()

print(f"\nMacro (positive-class only): P={macro_p:.4f}  R={macro_r:.4f}  F1={macro_f1:.4f}")
results_df

Rows used for evaluation: 961

OvR: beoordeling
                 precision    recall  f1-score   support

not_beoordeling     0.8618    0.8287    0.8449       572
    beoordeling     0.7616    0.8046    0.7825       389

       accuracy                         0.8189       961
      macro avg     0.8117    0.8166    0.8137       961
   weighted avg     0.8212    0.8189    0.8197       961


OvR: beslissing
                precision    recall  f1-score   support

not_beslissing     0.9922    0.9978    0.9950       892
    beslissing     0.9688    0.8986    0.9323        69

      accuracy                         0.9906       961
     macro avg     0.9805    0.9482    0.9637       961
  weighted avg     0.9905    0.9906    0.9905       961


OvR: materiele feiten
                      precision    recall  f1-score   support

not_materiele feiten     0.8618    0.9111    0.8858       664
    materiele feiten     0.7722    0.6734    0.7194       297

            accuracy                    

,label,precision,recall,f1,support
0,beoordeling,0.7616,0.8046,0.7825,389
1,beslissing,0.9688,0.8986,0.9323,69
2,materiele feiten,0.7722,0.6734,0.7194,297
3,proceshandelingen,0.6790,0.5340,0.5978,206


## 6. Header-prior Bayesian fusion

Same methodology as `01_five_way_inference.ipynb` section 8, applied independently to each of the 4 binary detectors (each gets its own header prior, built from the legal-labelled training subset, and its own tuned λ — there's no reason to assume the four detectors should share one weight).

In [6]:
from sklearn.metrics import precision_recall_fscore_support

TRAIN_DF_PATH = Path("predictions/train_df.csv")
EVAL_DF_PATH = Path("predictions/eval_df.csv")
ALPHA = 1.0
BINARY_CLASSES = [0, 1]

train_df = pd.read_csv(TRAIN_DF_PATH, keep_default_na=False)
eval_df = pd.read_csv(EVAL_DF_PATH, keep_default_na=False)
train_legal = train_df[train_df["label"].isin(LEGAL_LABELS)].copy()
eval_legal = eval_df[eval_df["label"].isin(LEGAL_LABELS)].copy()

print("Train rows (legal-labelled only):", len(train_legal))
print("Validation rows (legal-labelled only):", len(eval_legal))


def make_header_prior(df_train, label_col, labels, alpha=1.0):
    counts = (
        df_train.groupby(["hdr_group", label_col]).size()
        .unstack(fill_value=0)
        .reindex(columns=labels, fill_value=0)
    )
    probs = counts + alpha
    return probs.div(probs.sum(axis=1), axis=0)


def make_global_prior(df_train, label_col, labels, alpha=1.0):
    counts = df_train[label_col].value_counts().reindex(labels, fill_value=0) + alpha
    return (counts / counts.sum()).values


def get_meta_prior_df(df_apply, header_prior_train, global_prior, labels):
    def get_prior(hdr_group):
        if hdr_group in header_prior_train.index:
            return header_prior_train.loc[hdr_group].values
        return global_prior

    mat = np.vstack(df_apply["hdr_group"].apply(get_prior))
    return pd.DataFrame(mat, columns=labels, index=df_apply.index)


def combine_log_scores(text_probs, metadata_probs, lam, eps=1e-12):
    text = np.clip(text_probs.values, eps, 1.0)
    metadata = np.clip(metadata_probs.values, eps, 1.0)

    scores = np.log(text) + lam * np.log(metadata)
    scores -= scores.max(axis=1, keepdims=True)
    scores = np.exp(scores)
    scores /= scores.sum(axis=1, keepdims=True)

    return pd.DataFrame(scores, columns=text_probs.columns, index=text_probs.index)

Train rows (legal-labelled only): 3611
Validation rows (legal-labelled only): 759


In [7]:
LAMBDA_GRID = [0, 0.1, 0.25, 0.5, 1, 2, 3, 5]
legal_df = test_df[test_df["label"].isin(LEGAL_LABELS)].copy()

all_results = []
per_detector_sweeps = {}

for label in LEGAL_LABELS:
    clean = clean_col(label)
    model_key = f"ovr_{clean}"

    train_legal[f"target_{clean}"] = (train_legal["label"] == label).astype(int)
    eval_legal_target = (eval_legal["label"] == label).astype(int)

    # Re-run this detector on the validation set (models were deleted after section 3's loop)
    tokenizer, model, max_len = load_model(model_key)
    eval_probs = predict_probs(
        eval_legal[TEXT_COL].fillna("").astype(str).tolist(),
        tokenizer, model, max_len, BATCH_SIZE, desc=f"[val] {model_key}"
    )
    eval_text_probs = pd.DataFrame(eval_probs, columns=BINARY_CLASSES, index=eval_legal.index)
    del model, tokenizer
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    header_prior = make_header_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    global_prior = make_global_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    eval_meta_prior = get_meta_prior_df(eval_legal, header_prior, global_prior, BINARY_CLASSES)

    sweep = []
    for lam in LAMBDA_GRID:
        fused = combine_log_scores(eval_text_probs, eval_meta_prior, lam)
        pred = fused.idxmax(axis=1)
        _, _, f1, _ = precision_recall_fscore_support(
            eval_legal_target, pred, labels=BINARY_CLASSES, average="binary", pos_label=1, zero_division=0
        )
        sweep.append({"lambda": lam, "val_f1": round(f1, 4)})
    sweep_df = pd.DataFrame(sweep)
    best_lambda = sweep_df.loc[sweep_df["val_f1"].idxmax(), "lambda"]
    per_detector_sweeps[label] = sweep_df

    # Apply to test
    test_text_probs = legal_df[[f"{clean}_p_not", f"{clean}_p_pos"]].copy()
    test_text_probs.columns = BINARY_CLASSES
    test_meta_prior = get_meta_prior_df(legal_df, header_prior, global_prior, BINARY_CLASSES)
    test_target = (legal_df["label"] == label).astype(int)

    for lam, setting_type, name in [
        (0, "no_fusion", "no fusion"),
        (1, "lambda_1", "lambda=1"),
        (best_lambda, "tuned", f"tuned ({best_lambda})")
    ]:
        fused_test = combine_log_scores(test_text_probs, test_meta_prior, lam)
        pred_test = fused_test.idxmax(axis=1)
        precision, recall, f1, _ = precision_recall_fscore_support(
            test_target, pred_test, labels=BINARY_CLASSES, average="binary", pos_label=1, zero_division=0
        )
        all_results.append({
            "label": label, "setting": name, "setting_type": setting_type, "lambda": lam,
            "precision": round(precision, 4), "recall": round(recall, 4), "f1": round(f1, 4)
        })

results_df = pd.DataFrame(all_results)
results_df

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_beoordeling:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_beslissing:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_materiele_feiten:   0%|          | 0/24 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[val] ovr_proceshandelingen:   0%|          | 0/24 [00:00<?, ?it/s]

,label,setting,setting_type,lambda,precision,recall,f1
0,beoordeling,no fusion,no_fusion,0.0,0.7616,0.8046,0.7825
1,beoordeling,lambda=1,lambda_1,1.0,0.8015,0.7995,0.8005
2,beoordeling,tuned (2.0),tuned,2.0,0.8175,0.7943,0.8057
3,beslissing,no fusion,no_fusion,0.0,0.9688,0.8986,0.9323
4,beslissing,lambda=1,lambda_1,1.0,0.9839,0.8841,0.9313
5,beslissing,tuned (0.0),tuned,0.0,0.9688,0.8986,0.9323
6,materiele feiten,no fusion,no_fusion,0.0,0.7722,0.6734,0.7194
7,materiele feiten,lambda=1,lambda_1,1.0,0.8282,0.6330,0.7176
8,materiele feiten,tuned (2.0),tuned,2.0,0.8918,0.5825,0.7047
9,proceshandelingen,no fusion,no_fusion,0.0,0.6790,0.5340,0.5978


### Macro summary (positive-class only, averaged across the 4 detectors)

In [8]:
macro_summary = (
    results_df.groupby("setting_type")[["precision", "recall", "f1"]]
    .mean()
    .round(4)
    .reindex(["no_fusion", "lambda_1", "tuned"])
)
print(macro_summary)

print("\nPer-detector tuned lambda:")
for label in LEGAL_LABELS:
    row = results_df[(results_df["label"] == label) & (results_df["setting_type"] == "tuned")].iloc[0]
    print(f"  {label}: lambda={row['lambda']}")

              precision  recall      f1
setting_type                           
no_fusion        0.7954  0.7276  0.7580
lambda_1         0.8564  0.7102  0.7716
tuned            0.8414  0.7024  0.7610

Per-detector tuned lambda:
  beoordeling: lambda=2.0
  beslissing: lambda=0.0
  materiele feiten: lambda=2.0
  proceshandelingen: lambda=0.1


In [9]:
from sklearn.metrics import classification_report

# The paper's "OvR mean" (app:full_supervised_results) uses the both-class average
# per detector (accuracy, macro P/R/F1 over {not-X, X}, weighted F1), then averages
# those four detectors' own stats — a different convention from the positive-class-only
# numbers above. Reusing each detector's own tuned lambda found in section 6.

tuned_lambdas = {
    label: results_df.loc[
        (results_df["label"] == label) & (results_df["setting_type"] == "tuned"), "lambda"
    ].iloc[0]
    for label in LEGAL_LABELS
}
print("Tuned lambda per detector:", tuned_lambdas)

per_detector_full = []
for label in LEGAL_LABELS:
    clean = clean_col(label)
    text_probs = legal_df[[f"{clean}_p_not", f"{clean}_p_pos"]].copy()
    text_probs.columns = BINARY_CLASSES

    header_prior = make_header_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    global_prior = make_global_prior(train_legal, f"target_{clean}", BINARY_CLASSES, ALPHA)
    meta_prior = get_meta_prior_df(legal_df, header_prior, global_prior, BINARY_CLASSES)

    fused = combine_log_scores(text_probs, meta_prior, tuned_lambdas[label])
    pred = fused.idxmax(axis=1)
    target = (legal_df["label"] == label).astype(int)

    rep = classification_report(target, pred, labels=BINARY_CLASSES, output_dict=True, zero_division=0)
    row = {
        "label": label,
        "accuracy": round(rep["accuracy"], 4),
        "macro_p": round(rep["macro avg"]["precision"], 4),
        "macro_r": round(rep["macro avg"]["recall"], 4),
        "macro_f1": round(rep["macro avg"]["f1-score"], 4),
        "weighted_f1": round(rep["weighted avg"]["f1-score"], 4),
    }
    per_detector_full.append(row)
    print(f"{label} (both-class, tuned lambda={tuned_lambdas[label]}): {row}")

per_detector_df = pd.DataFrame(per_detector_full)
ovr_mean_both_class = per_detector_df[["accuracy", "macro_p", "macro_r", "macro_f1", "weighted_f1"]].mean().round(4)
print("\nOvR mean (both-class average across 4 detectors, tuned lambda):")
print(ovr_mean_both_class)

Tuned lambda per detector: {'beoordeling': 2.0, 'beslissing': 0.0, 'materiele feiten': 2.0, 'proceshandelingen': 0.1}
beoordeling (both-class, tuned lambda=2.0): {'label': 'beoordeling', 'accuracy': 0.845, 'macro_p': 0.8401, 'macro_r': 0.8369, 'macro_f1': 0.8384, 'weighted_f1': 0.8446}
beslissing (both-class, tuned lambda=0.0): {'label': 'beslissing', 'accuracy': 0.9906, 'macro_p': 0.9805, 'macro_r': 0.9482, 'macro_f1': 0.9637, 'weighted_f1': 0.9905}
materiele feiten (both-class, tuned lambda=2.0): {'label': 'materiele feiten', 'accuracy': 0.8491, 'macro_p': 0.865, 'macro_r': 0.7754, 'macro_f1': 0.8017, 'weighted_f1': 0.8387}
proceshandelingen (both-class, tuned lambda=0.1): {'label': 'proceshandelingen', 'accuracy': 0.8481, 'macro_p': 0.7838, 'macro_r': 0.7339, 'macro_f1': 0.7536, 'weighted_f1': 0.8408}

OvR mean (both-class average across 4 detectors, tuned lambda):
accuracy       0.8832
macro_p        0.8674
macro_r        0.8236
macro_f1       0.8394
weighted_f1    0.8786
dtype: fl